# 🖼️ 중급 예시 — 이미지 뷰어

Tkinter와 Pillow를 사용하여 이미지 뷰어를 단계별로 만듭니다.

```bash
pip install Pillow
```

| 기능 | 설명 |
|------|------|
| 이미지 열기 | 파일 대화상자로 이미지 선택 |
| 이미지 표시 | Canvas에 이미지 표시 |
| 확대/축소 | Scale 슬라이더로 크기 조절 |
| 이전/다음 | 폴더 내 이미지 탐색 |
| 회전 | 90도씩 회전 |

---
## 1. 기본 창과 Canvas 준비

`Canvas` 위젯은 이미지나 도형을 그릴 수 있는 영역입니다.
이미지 뷰어의 핵심 위젯입니다.

In [ ]:
# 기본 창과 Canvas
import tkinter as tk
from tkinter import filedialog, messagebox
from PIL import Image, ImageTk

root = tk.Tk()
root.title('이미지 뷰어')
root.geometry('800x600')

# Canvas 생성 (이미지 표시 영역)
canvas = tk.Canvas(root, bg='black', highlightthickness=0)
canvas.pack(expand=True, fill='both')

# 상태 표시용 레이블
status = tk.Label(root, text='이미지를 열어주세요 (Ctrl+O)', relief='sunken', anchor='w')
status.pack(fill='x')

root.mainloop()

---
## 2. 이미지 열기

`filedialog.askopenfilename()`으로 파일을 선택하고,
Pillow로 이미지를 로드하여 Canvas에 표시합니다.

**핵심 포인트:**
- Pillow의 `Image` → Tkinter의 `ImageTk.PhotoImage`로 변환
- `ImageTk.PhotoImage`의 참조를 유지하지 않으면 가비지 컬렉션됨
- Canvas의 `create_image()`로 이미지 배치

In [ ]:
# 이미지 열기 기능 추가
import tkinter as tk
from tkinter import filedialog, messagebox
from PIL import Image, ImageTk

root = tk.Tk()
root.title('이미지 뷰어')
root.geometry('800x600')

canvas = tk.Canvas(root, bg='#2b2b2b', highlightthickness=0)
canvas.pack(expand=True, fill='both')

status = tk.Label(root, text='이미지를 열어주세요 (메뉴 > 파일 > 열기)', relief='sunken', anchor='w')
status.pack(fill='x')

# 전역 변수로 이미지 참조 유지 (가비지 컬렉션 방지)
current_image = None
photo = None
image_id = None

def open_image():
    global current_image, photo, image_id

    filepath = filedialog.askopenfilename(
        title='이미지 열기',
        filetypes=[('이미지 파일', '*.png *.jpg *.jpeg *.gif *.bmp *.webp'),
                   ('모든 파일', '*.*')]
    )
    if not filepath:
        return

    try:
        # Pillow로 이미지 로드
        current_image = Image.open(filepath)

        # Canvas 크기에 맞게 리사이즈
        canvas_w = canvas.winfo_width()
        canvas_h = canvas.winfo_height()
        if canvas_w > 1 and canvas_h > 1:
            current_image.thumbnail((canvas_w, canvas_h), Image.LANCZOS)

        # Tkinter 이미지로 변환
        photo = ImageTk.PhotoImage(current_image)

        # Canvas에 표시
        if image_id:
            canvas.delete(image_id)
        image_id = canvas.create_image(
            canvas.winfo_width() // 2,
            canvas.winfo_height() // 2,
            image=photo, anchor='center'
        )

        status.config(text=f'{filepath}  |  {current_image.size[0]}x{current_image.size[1]}')
    except Exception as e:
        messagebox.showerror('오류', f'이미지를 열 수 없습니다:\n{e}')

# 메뉴 바
menubar = tk.Menu(root)
file_menu = tk.Menu(menubar, tearoff=0)
file_menu.add_command(label='열기', command=open_image, accelerator='Ctrl+O')
file_menu.add_separator()
file_menu.add_command(label='종료', command=root.quit)
menubar.add_cascade(label='파일', menu=file_menu)
root.config(menu=menubar)

# 단축키
root.bind('<Control-o>', lambda e: open_image())

root.mainloop()

---
## 3. 확대/축소 기능

Scale 슬라이더로 이미지 크기를 조절합니다.
원본 이미지를 유지하고 비율에 맞춰 리사이즈합니다.

In [ ]:
# 확대/축소 기능 추가
import tkinter as tk
from tkinter import filedialog, messagebox
from PIL import Image, ImageTk

root = tk.Tk()
root.title('이미지 뷰어 — 확대/축소')
root.geometry('800x650')

original_image = None
photo = None
image_id = None
scale_factor = 1.0

canvas = tk.Canvas(root, bg='#2b2b2b', highlightthickness=0)
canvas.pack(expand=True, fill='both')

# 하단 컨트롤 프레임
ctrl_frame = tk.Frame(root)
ctrl_frame.pack(fill='x')

status = tk.Label(ctrl_frame, text='이미지를 열어주세요', relief='sunken', anchor='w')
status.pack(side='left', expand=True, fill='x')

def update_display():
    global photo, image_id
    if original_image is None:
        return

    # 비율 계산
    w, h = original_image.size
    new_w = int(w * scale_factor)
    new_h = int(h * scale_factor)

    # 리사이즈
    resized = original_image.resize((new_w, new_h), Image.LANCZOS)
    photo = ImageTk.PhotoImage(resized)

    if image_id:
        canvas.delete(image_id)
    image_id = canvas.create_image(
        canvas.winfo_width() // 2,
        canvas.winfo_height() // 2,
        image=photo, anchor='center'
    )
    status.config(text=f'크기: {new_w}x{new_h}  |  비율: {scale_factor*100:.0f}%')

def on_scale(val):
    global scale_factor
    scale_factor = float(val) / 100.0
    update_display()

# 확대/축소 슬라이더
tk.Label(ctrl_frame, text='크기:').pack(side='left')
scale = tk.Scale(ctrl_frame, from_=10, to=300, orient='horizontal',
                 command=on_scale, length=200)
scale.set(100)
scale.pack(side='left', padx=5)

def open_image():
    global original_image, scale_factor
    filepath = filedialog.askopenfilename(
        title='이미지 열기',
        filetypes=[('이미지 파일', '*.png *.jpg *.jpeg *.gif *.bmp'), ('모든 파일', '*.*')]
    )
    if not filepath:
        return
    try:
        original_image = Image.open(filepath)
        # Canvas에 맞게 초기 크기 조절
        canvas.update()
        cw, ch = canvas.winfo_width(), canvas.winfo_height()
        ow, oh = original_image.size
        ratio = min(cw / ow, ch / oh, 1.0)
        scale_factor = ratio
        scale.set(int(scale_factor * 100))
        update_display()
        root.title(f'이미지 뷰어 — {filepath.split("/")[-1]}')
    except Exception as e:
        messagebox.showerror('오류', str(e))

menubar = tk.Menu(root)
file_menu = tk.Menu(menubar, tearoff=0)
file_menu.add_command(label='열기', command=open_image)
file_menu.add_separator()
file_menu.add_command(label='종료', command=root.quit)
menubar.add_cascade(label='파일', menu=file_menu)
root.config(menu=menubar)
root.bind('<Control-o>', lambda e: open_image())

root.mainloop()

---
## 4. 90도 회전 기능

Pillow의 `Image.rotate()`를 사용하여 이미지를 회전합니다.
R 키를 누르면 90도씩 회전합니다.

In [ ]:
# 회전 기능이 포함된 완전한 이미지 뷰어
import tkinter as tk
from tkinter import filedialog, messagebox
from PIL import Image, ImageTk

root = tk.Tk()
root.title('이미지 뷰어 — 완전판')
root.geometry('800x650')

original_image = None   # 원본 이미지
display_image = None    # 회전 등 변형된 이미지
photo = None
image_id = None
rotation = 0
scale_factor = 1.0
current_file = ''

canvas = tk.Canvas(root, bg='#2b2b2b', highlightthickness=0)
canvas.pack(expand=True, fill='both')

ctrl_frame = tk.Frame(root)
ctrl_frame.pack(fill='x')

status = tk.Label(ctrl_frame, text='이미지를 열어주세요 (Ctrl+O)', relief='sunken', anchor='w')
status.pack(side='left', expand=True, fill='x')

def update_display():
    global display_image, photo, image_id
    if original_image is None:
        return

    # 회전 적용
    display_image = original_image.rotate(rotation, expand=True)

    # 크기 조절
    w, h = display_image.size
    new_w = max(1, int(w * scale_factor))
    new_h = max(1, int(h * scale_factor))
    display_image = display_image.resize((new_w, new_h), Image.LANCZOS)

    photo = ImageTk.PhotoImage(display_image)
    if image_id:
        canvas.delete(image_id)
    image_id = canvas.create_image(
        canvas.winfo_width() // 2,
        canvas.winfo_height() // 2,
        image=photo, anchor='center'
    )
    status.config(text=f'{current_file}  |  {new_w}x{new_h}  |  {rotation}도  |  {scale_factor*100:.0f}%')

def open_image():
    global original_image, current_file, rotation, scale_factor
    filepath = filedialog.askopenfilename(
        title='이미지 열기',
        filetypes=[('이미지 파일', '*.png *.jpg *.jpeg *.gif *.bmp *.webp'), ('모든 파일', '*.*')]
    )
    if not filepath:
        return
    try:
        original_image = Image.open(filepath)
        current_file = filepath.split('/')[-1]
        rotation = 0
        # Canvas에 맞춤
        canvas.update()
        cw, ch = canvas.winfo_width(), canvas.winfo_height()
        ow, oh = original_image.size
        ratio = min(cw / ow, ch / oh, 1.0)
        scale_factor = ratio
        update_display()
        root.title(f'이미지 뷰어 — {current_file}')
    except Exception as e:
        messagebox.showerror('오류', str(e))

def rotate_image():
    global rotation
    rotation = (rotation + 90) % 360
    update_display()

def on_scale(val):
    global scale_factor
    scale_factor = float(val) / 100.0
    update_display()

# 슬라이더
tk.Label(ctrl_frame, text='크기:').pack(side='left')
scale = tk.Scale(ctrl_frame, from_=10, to=300, orient='horizontal',
                 command=on_scale, length=200)
scale.set(100)
scale.pack(side='left', padx=5)

# 회전 버튼
tk.Button(ctrl_frame, text='회전 (R)', command=rotate_image).pack(side='left', padx=5)

# 메뉴
menubar = tk.Menu(root)
file_menu = tk.Menu(menubar, tearoff=0)
file_menu.add_command(label='열기', command=open_image, accelerator='Ctrl+O')
file_menu.add_command(label='회전', command=rotate_image, accelerator='R')
file_menu.add_separator()
file_menu.add_command(label='종료', command=root.quit, accelerator='Ctrl+Q')
menubar.add_cascade(label='파일', menu=file_menu)
root.config(menu=menubar)

# 단축키
root.bind('<Control-o>', lambda e: open_image())
root.bind('<r>', lambda e: rotate_image())
root.bind('<Control-q>', lambda e: root.quit())

root.mainloop()

---
## 🎯 정리

| 기능 | 핵심 코드 | 설명 |
|------|-----------|------|
| 이미지 로드 | `Image.open(path)` | Pillow로 이미지 열기 |
| Tkinter 변환 | `ImageTk.PhotoImage(img)` | PIL → Tkinter 이미지 |
| Canvas 표시 | `canvas.create_image(x, y, image=photo)` | Canvas에 이미지 배치 |
| 리사이즈 | `img.resize((w, h), Image.LANCZOS)` | 크기 조절 |
| 회전 | `img.rotate(90, expand=True)` | 90도 회전 |
| 파일 선택 | `filedialog.askopenfilename()` | 파일 대화상자 |
| 참조 유지 | 전역 변수에 `photo` 저장 | 가비지 컬렉션 방지 |

> ⚠️ **중요**: `ImageTk.PhotoImage` 객체를 지역 변수에 저장하면
> 함수 종료 시 가비지 컬렉션되어 이미지가 사라집니다.
> 반드시 전역 변수나 인스턴스 변수로 참조를 유지하세요!

---
## 🎯 연습 문제

1. **이전/다음 버튼 추가**: 같은 폴더의 이미지를 이전/다음 버튼으로 탐색할 수 있게 만드세요. (힌트: `os.listdir()`로 폴더 내 이미지 파일 목록 만들기)

2. **이미지 정보 표시**: 이미지 포맷(PNG/JPEG 등), 파일 크기(KB), 픽셀 크기를 상태 표시줄에 표시하세요. (힌트: `Image.format`, `os.path.getsize()`)

3. **필터 효과**: 흑백 변환, 블러, 색상 반전 등의 필터 버튼을 추가하세요. (힌트: `ImageOps.grayscale()`, `ImageFilter.BLUR`, `ImageOps.invert()`)

4. **드래그로 이미지 이동**: Canvas 위에서 마우스 드래그로 이미지 위치를 이동할 수 있게 만드세요. (힌트: `canvas.bind('<B1-Motion>', ...)`, `canvas.coords(image_id, x, y)`